# Comprehensive Heart Rate Variability (HRV) Analysis

This notebook provides a complete, comprehensive analysis of HRV using the best methods available in Python.

## Overview

This analysis includes:
- **Time Domain Metrics**: SDNN, RMSSD, pNN50, etc.
- **Frequency Domain Metrics**: VLF, LF, HF powers, LF/HF ratio
- **Nonlinear Metrics**: Poincaré plot, DFA, entropy measures
- **Autonomic Nervous System Analysis**: Parasympathetic and sympathetic indices
- **Advanced Statistical Measures**: Confidence intervals, quality assessment

## Requirements

The following Python packages are required:
- `numpy` - Numerical computations
- `pandas` - Data manipulation
- `scipy` - Scientific computing and signal processing
- `matplotlib` - Plotting
- `seaborn` - Statistical visualization
- `hrvanalysis` - HRV analysis library (optional, for additional metrics)

## Data Format

Input data can be provided in multiple formats:

1. **RR Intervals Text File**: A text file with one RR interval (in milliseconds) per line
   - Example: `2025-11-06 00-50-07.txt`
   - The notebook will automatically load and process these RR intervals

2. **CSV File with Heart Rate**: A pandas DataFrame with:
   - Heart rate column: `heart_rate [bpm]` or similar
   - Optional grouping columns: `subject`, `Sol`, `condition`, etc.
   - Optional time column: `timestamp` or `time [s/1000]`

3. **CSV File with RR Intervals**: A pandas DataFrame with:
   - RR intervals column: `rr_intervals_ms` (in milliseconds)
   - Optional grouping columns: `subject`, `Sol`, `condition`, etc.


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.interpolate import interp1d
from typing import Dict, Any, Optional, Tuple, List
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Try to import hrvanalysis library (optional)
try:
    from hrvanalysis import (
        get_time_domain_features,
        get_frequency_domain_features,
        get_poincare_plot_features,
    )
    HAS_HRVANALYSIS = True
    print("✅ hrvanalysis library available")
except ImportError:
    HAS_HRVANALYSIS = False
    print("⚠️ hrvanalysis library not available - using custom implementations")

print("Libraries imported successfully!")


: 

## 1. Data Loading and Preparation

Load your data from CSV files, database, or create sample data for testing.


In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

def load_rr_intervals_from_txt(file_path: str) -> np.ndarray:
    """
    Load RR intervals from a text file with one value per line (in milliseconds).
    
    Args:
        file_path: Path to the text file containing RR intervals
    
    Returns:
        Array of RR intervals in milliseconds
    """
    try:
        with open(file_path, 'r') as f:
            lines = f.readlines()
        
        rr_intervals = []
        for line in lines:
            line = line.strip()
            if line:  # Skip empty lines
                try:
                    rr_value = float(line)
                    # Filter physiologically plausible values (300-2000 ms)
                    if 300 <= rr_value <= 2000:
                        rr_intervals.append(rr_value)
                except ValueError:
                    continue  # Skip non-numeric lines
        
        rr_array = np.array(rr_intervals)
        print(f"✅ Loaded {len(rr_array):,} RR intervals from {file_path}")
        print(f"   RR interval range: {rr_array.min():.1f} - {rr_array.max():.1f} ms")
        print(f"   Mean RR: {rr_array.mean():.1f} ms ({60000/rr_array.mean():.1f} bpm)")
        return rr_array
        
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return np.array([])
    except Exception as e:
        print(f"❌ Error loading RR intervals: {e}")
        return np.array([])

def load_data_from_csv(file_path: str) -> pd.DataFrame:
    """Load data from CSV file."""
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Loaded {len(df):,} rows from {file_path}")
        return df
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

def load_multiple_csvs(file_paths: Dict[str, str]) -> pd.DataFrame:
    """Load and combine multiple CSV files."""
    dfs = {}
    for name, path in file_paths.items():
        try:
            dfs[name] = pd.read_csv(path)
            dfs[name]['source'] = name
            print(f"✅ Loaded {name}: {len(dfs[name]):,} rows")
        except Exception as e:
            print(f"❌ Error loading {name}: {e}")
    
    if dfs:
        df_combined = pd.concat(dfs.values(), ignore_index=True)
        print(f"✅ Combined dataset: {len(df_combined):,} rows")
        return df_combined
    return None

# =============================================================================
# LOAD RR INTERVALS FROM TEXT FILE
# =============================================================================

# Load RR intervals from the specified file
rr_file_path = '2025-11-06 00-50-07.txt'
rr_intervals = load_rr_intervals_from_txt(rr_file_path)

if len(rr_intervals) > 0:
    # Convert RR intervals to heart rate for compatibility with existing functions
    heart_rate_bpm = 60000.0 / rr_intervals
    
    # Create DataFrame with RR intervals and heart rate
    df = pd.DataFrame({
        'rr_intervals_ms': rr_intervals,
        'heart_rate [bpm]': heart_rate_bpm,
        'beat_index': np.arange(len(rr_intervals)),
        'timestamp': pd.date_range('2025-11-06 00:50:07', periods=len(rr_intervals), 
                                   freq=pd.Timedelta(milliseconds=rr_intervals.mean()))
    })
    
    print(f"\n✅ Created DataFrame with {len(df):,} beats")
    print(f"   Recording duration: {rr_intervals.sum() / 1000 / 60:.2f} minutes")
else:
    print("\n⚠️ No RR intervals loaded. Creating sample data for demonstration...")
    np.random.seed(42)
    n_samples = 1000
    base_hr = 70
    noise = np.random.normal(0, 5, n_samples)
    trend = np.sin(np.linspace(0, 4*np.pi, n_samples)) * 10
    
    heart_rate = base_hr + noise + trend
    heart_rate = np.clip(heart_rate, 50, 120)
    
    df = pd.DataFrame({
        'heart_rate [bpm]': heart_rate,
        'subject': ['SampleSubject'] * n_samples,
        'Sol': [1] * n_samples,
        'timestamp': pd.date_range('2023-01-01', periods=n_samples, freq='1s')
    })
    print(f"✅ Created sample dataset with {len(df):,} rows")

# Display data info
print("\nData Info:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


## 2. Core HRV Analysis Functions

These functions implement the best HRV analysis methods available in Python.


In [ ]:
# =============================================================================
# CORE HRV ANALYSIS FUNCTIONS
# =============================================================================

def rr_from_hr(hr_series: pd.Series, min_rr: float = 300, max_rr: float = 2000) -> np.ndarray:
    """
    Convert heart rate (bpm) to RR intervals (ms).
    
    Args:
        hr_series: Heart rate data in BPM
        min_rr: Minimum physiologically plausible RR interval (ms)
        max_rr: Maximum physiologically plausible RR interval (ms)
    
    Returns:
        Array of RR intervals in milliseconds
    """
    try:
        hr_clean = pd.to_numeric(hr_series, errors='coerce').dropna()
        
        if len(hr_clean) == 0:
            return np.array([])
        
        # Convert to RR intervals: RR(ms) = 60000 / HR(bpm)
        rr_ms = 60000.0 / hr_clean
        rr_ms = rr_ms.replace([np.inf, -np.inf], np.nan).dropna()
        
        # Filter physiologically plausible values
        rr_ms = rr_ms[(rr_ms >= min_rr) & (rr_ms <= max_rr)]
        
        return rr_ms.values
        
    except Exception as e:
        print(f"❌ Error converting HR to RR: {e}")
        return np.array([])

def compute_time_domain_metrics(rr_intervals: np.ndarray) -> Dict[str, float]:
    """
    Compute time domain HRV metrics.
    
    Returns:
        Dictionary of time domain metrics
    """
    if len(rr_intervals) == 0:
        return {}
    
    metrics = {}
    
    # Basic statistics
    metrics['mean_nni'] = np.mean(rr_intervals)
    metrics['sdnn'] = np.std(rr_intervals, ddof=1)
    metrics['median_nni'] = np.median(rr_intervals)
    metrics['mad_nni'] = np.median(np.abs(rr_intervals - metrics['median_nni']))
    metrics['cvnn'] = (metrics['sdnn'] / metrics['mean_nni']) * 100 if metrics['mean_nni'] > 0 else 0
    
    # Heart rate statistics
    hr_values = 60000 / rr_intervals
    metrics['mean_hr'] = np.mean(hr_values)
    metrics['std_hr'] = np.std(hr_values, ddof=1)
    metrics['min_hr'] = np.min(hr_values)
    metrics['max_hr'] = np.max(hr_values)
    
    # Successive difference measures
    if len(rr_intervals) > 1:
        rr_diff = np.diff(rr_intervals)
        metrics['rmssd'] = np.sqrt(np.mean(rr_diff ** 2))
        metrics['sdsd'] = np.std(rr_diff, ddof=1)
        metrics['cvsd'] = (metrics['sdsd'] / np.mean(np.abs(rr_diff))) * 100 if np.mean(np.abs(rr_diff)) > 0 else 0
        
        # NN50 and pNN50
        nn50 = np.sum(np.abs(rr_diff) > 50)
        metrics['nn50'] = int(nn50)
        metrics['pnn50'] = (nn50 / len(rr_diff)) * 100
        
        # NN20 and pNN20
        nn20 = np.sum(np.abs(rr_diff) > 20)
        metrics['nn20'] = int(nn20)
        metrics['pnn20'] = (nn20 / len(rr_diff)) * 100
    else:
        metrics['rmssd'] = 0
        metrics['sdsd'] = 0
        metrics['cvsd'] = 0
        metrics['nn50'] = 0
        metrics['pnn50'] = 0
        metrics['nn20'] = 0
        metrics['pnn20'] = 0
    
    return metrics

print("✅ Core HRV functions defined")


In [ ]:
def compute_frequency_domain_metrics(rr_intervals: np.ndarray, 
                                     method: str = 'welch',
                                     sampling_rate: float = 4.0) -> Dict[str, float]:
    """
    Compute frequency domain HRV metrics using power spectral density.
    
    Args:
        rr_intervals: RR intervals in milliseconds
        method: PSD estimation method ('welch' or 'periodogram')
        sampling_rate: Sampling rate for interpolation (Hz)
    
    Returns:
        Dictionary of frequency domain metrics
    """
    if len(rr_intervals) < 50:
        return {}
    
    try:
        # Create time series for interpolation
        rr_seconds = rr_intervals / 1000.0
        r_peak_times = np.concatenate([[0], np.cumsum(rr_seconds)])
        rr_timestamps = (r_peak_times[:-1] + r_peak_times[1:]) / 2.0
        
        # Create regular time grid
        total_duration = r_peak_times[-1]
        if total_duration <= 0:
            return {}
        
        time_regular = np.arange(0, total_duration, 1/sampling_rate)
        
        if len(time_regular) < 10:
            return {}
        
        # Interpolate RR intervals
        interp_kind = 'cubic' if len(rr_intervals) >= 4 else 'linear'
        f_interp = interp1d(rr_timestamps, rr_intervals, 
                          kind=interp_kind,
                          bounds_error=False, 
                          fill_value='extrapolate')
        rr_interpolated = f_interp(time_regular)
        
        # Detrend
        rr_detrended = signal.detrend(rr_interpolated)
        
        # Compute power spectral density
        if method == 'welch':
            freqs, psd = signal.welch(rr_detrended, fs=sampling_rate, 
                                    nperseg=min(len(rr_detrended)//4, 256),
                                    window='hann')
        else:
            freqs, psd = signal.periodogram(rr_detrended, fs=sampling_rate, window='hann')
        
        # Define frequency bands (Hz)
        vlf_band = (0.0033, 0.04)
        lf_band = (0.04, 0.15)
        hf_band = (0.15, 0.4)
        
        # Calculate power in each band
        vlf_mask = (freqs >= vlf_band[0]) & (freqs < vlf_band[1])
        lf_mask = (freqs >= lf_band[0]) & (freqs < lf_band[1])
        hf_mask = (freqs >= hf_band[0]) & (freqs < hf_band[1])
        
        vlf_power = np.trapz(psd[vlf_mask], freqs[vlf_mask]) if np.any(vlf_mask) else 0
        lf_power = np.trapz(psd[lf_mask], freqs[lf_mask]) if np.any(lf_mask) else 0
        hf_power = np.trapz(psd[hf_mask], freqs[hf_mask]) if np.any(hf_mask) else 0
        total_power = vlf_power + lf_power + hf_power
        
        # Normalized units
        lf_nu = (lf_power / (lf_power + hf_power)) * 100 if (lf_power + hf_power) > 0 else 0
        hf_nu = (hf_power / (lf_power + hf_power)) * 100 if (lf_power + hf_power) > 0 else 0
        lf_hf_ratio = lf_power / hf_power if hf_power > 0 else 0
        
        # Percentages
        vlf_percent = (vlf_power / total_power) * 100 if total_power > 0 else 0
        lf_percent = (lf_power / total_power) * 100 if total_power > 0 else 0
        hf_percent = (hf_power / total_power) * 100 if total_power > 0 else 0
        
        # Peak frequencies
        vlf_peak = freqs[vlf_mask][np.argmax(psd[vlf_mask])] if np.any(vlf_mask) and vlf_power > 0 else 0
        lf_peak = freqs[lf_mask][np.argmax(psd[lf_mask])] if np.any(lf_mask) and lf_power > 0 else 0
        hf_peak = freqs[hf_mask][np.argmax(psd[hf_mask])] if np.any(hf_mask) and hf_power > 0 else 0
        
        return {
            'vlf_power': vlf_power,
            'lf_power': lf_power,
            'hf_power': hf_power,
            'total_power': total_power,
            'lf_nu': lf_nu,
            'hf_nu': hf_nu,
            'lf_hf_ratio': lf_hf_ratio,
            'vlf_percent': vlf_percent,
            'lf_percent': lf_percent,
            'hf_percent': hf_percent,
            'vlf_peak': vlf_peak,
            'lf_peak': lf_peak,
            'hf_peak': hf_peak,
            'method': method,
            'sampling_rate': sampling_rate
        }
        
    except Exception as e:
        print(f"⚠️ Frequency domain analysis failed: {e}")
        return {}

print("✅ Frequency domain functions defined")


In [ ]:
def compute_poincare_metrics(rr_intervals: np.ndarray) -> Dict[str, float]:
    """
    Compute Poincaré plot metrics.
    
    Returns:
        Dictionary of Poincaré plot metrics
    """
    if len(rr_intervals) < 2:
        return {}
    
    try:
        rr1 = rr_intervals[:-1]
        rr2 = rr_intervals[1:]
        
        # SD1 and SD2
        diff = rr2 - rr1
        sum_rr = rr2 + rr1
        
        sd1 = np.std(diff) / np.sqrt(2)
        sd2 = np.std(sum_rr) / np.sqrt(2)
        sd1_sd2_ratio = sd2 / sd1 if sd1 > 0 else 0
        
        # Ellipse area
        ellipse_area = np.pi * sd1 * sd2 if sd1 > 0 and sd2 > 0 else 0
        
        return {
            'sd1': sd1,
            'sd2': sd2,
            'sd1_sd2_ratio': sd1_sd2_ratio,
            'ellipse_area': ellipse_area
        }
        
    except Exception as e:
        print(f"⚠️ Poincaré analysis failed: {e}")
        return {}

def compute_dfa_metrics(rr_intervals: np.ndarray) -> Dict[str, float]:
    """
    Compute Detrended Fluctuation Analysis (DFA) metrics.
    
    Returns:
        Dictionary with DFA alpha1 and alpha2
    """
    if len(rr_intervals) < 100:
        return {'dfa_alpha1': 0, 'dfa_alpha2': 0}
    
    try:
        # Create cumulative sum
        y = np.cumsum(rr_intervals - np.mean(rr_intervals))
        
        # Define scales
        scales_short = np.arange(4, min(17, len(rr_intervals)//4))
        scales_long = np.arange(16, min(65, len(rr_intervals)//4))
        
        def compute_fluctuation(scales):
            fluctuations = []
            for scale in scales:
                segments = len(y) // scale
                if segments < 4:
                    continue
                
                local_fluctuations = []
                for i in range(segments):
                    start = i * scale
                    end = start + scale
                    segment = y[start:end]
                    
                    # Detrend
                    x_segment = np.arange(len(segment))
                    coeffs = np.polyfit(x_segment, segment, 1)
                    trend = np.polyval(coeffs, x_segment)
                    detrended = segment - trend
                    
                    local_fluctuations.append(np.sqrt(np.mean(detrended**2)))
                
                if local_fluctuations:
                    fluctuations.append(np.mean(local_fluctuations))
            
            return fluctuations
        
        fluc_short = compute_fluctuation(scales_short)
        fluc_long = compute_fluctuation(scales_long)
        
        # Calculate scaling exponents
        alpha1 = alpha2 = 0
        
        if len(fluc_short) > 2 and np.all(np.array(fluc_short) > 0):
            log_scales = np.log10(scales_short[:len(fluc_short)])
            log_fluc = np.log10(fluc_short)
            alpha1, _ = np.polyfit(log_scales, log_fluc, 1)
        
        if len(fluc_long) > 2 and np.all(np.array(fluc_long) > 0):
            log_scales = np.log10(scales_long[:len(fluc_long)])
            log_fluc = np.log10(fluc_long)
            alpha2, _ = np.polyfit(log_scales, log_fluc, 1)
        
        return {
            'dfa_alpha1': float(alpha1),
            'dfa_alpha2': float(alpha2)
        }
        
    except Exception as e:
        print(f"⚠️ DFA computation failed: {e}")
        return {'dfa_alpha1': 0, 'dfa_alpha2': 0}

print("✅ Nonlinear metrics functions defined")


In [ ]:
def compute_comprehensive_hrv(rr_intervals: np.ndarray, 
                              use_hrvanalysis: bool = False) -> Dict[str, Any]:
    """
    Compute comprehensive HRV metrics using all available methods.
    
    Args:
        rr_intervals: RR intervals in milliseconds
        use_hrvanalysis: Whether to use hrvanalysis library if available
    
    Returns:
        Dictionary containing all HRV metrics
    """
    results = {
        'n_intervals': len(rr_intervals),
        'recording_duration_minutes': np.sum(rr_intervals) / (1000 * 60)
    }
    
    if len(rr_intervals) < 10:
        return results
    
    # Time domain metrics
    time_metrics = compute_time_domain_metrics(rr_intervals)
    results.update(time_metrics)
    
    # Frequency domain metrics
    freq_metrics = compute_frequency_domain_metrics(rr_intervals)
    results.update(freq_metrics)
    
    # Poincaré metrics
    poincare_metrics = compute_poincare_metrics(rr_intervals)
    results.update(poincare_metrics)
    
    # DFA metrics
    dfa_metrics = compute_dfa_metrics(rr_intervals)
    results.update(dfa_metrics)
    
    # Use hrvanalysis library if available and requested
    if use_hrvanalysis and HAS_HRVANALYSIS and len(rr_intervals) >= 50:
        try:
            hrv_time = get_time_domain_features(rr_intervals)
            hrv_freq = get_frequency_domain_features(rr_intervals)
            hrv_poincare = get_poincare_plot_features(rr_intervals)
            
            # Add prefix to avoid conflicts
            results.update({f'hrvanalysis_{k}': v for k, v in hrv_time.items()})
            results.update({f'hrvanalysis_{k}': v for k, v in hrv_freq.items()})
            results.update({f'hrvanalysis_{k}': v for k, v in hrv_poincare.items()})
        except Exception as e:
            print(f"⚠️ hrvanalysis library computation failed: {e}")
    
    # Compute ANS indices
    if 'hf_power' in results and 'rmssd' in results:
        # Parasympathetic index (higher HF, RMSSD, pNN50, SD1 = higher parasympathetic)
        parasympathetic_components = [
            results.get('hf_nu', 0) / 100,
            min(1.0, results.get('rmssd', 0) / 100),
            min(1.0, results.get('pnn50', 0) / 50),
            min(1.0, results.get('sd1', 0) / 50)
        ]
        results['parasympathetic_index'] = np.mean([c for c in parasympathetic_components if c > 0])
        
        # Sympathetic index (higher LF/HF ratio = higher sympathetic)
        lf_hf_ratio = results.get('lf_hf_ratio', 0)
        results['sympathetic_index'] = min(1.0, lf_hf_ratio / 5.0)
        
        # ANS balance
        results['ans_balance'] = results['parasympathetic_index'] - results['sympathetic_index']
    
    return results

print("✅ Comprehensive HRV analysis function defined")


## 3. Perform HRV Analysis

Analyze HRV for your data. You can analyze the entire dataset or group by subject, condition, etc.


In [ ]:
# =============================================================================
# PERFORM HRV ANALYSIS
# =============================================================================

def analyze_hrv_dataframe(df: pd.DataFrame, 
                         hr_column: str = 'heart_rate [bpm]',
                         rr_column: str = 'rr_intervals_ms',
                         group_by: List[str] = None) -> pd.DataFrame:
    """
    Analyze HRV for a DataFrame, optionally grouped by columns.
    
    Args:
        df: Input DataFrame
        hr_column: Name of heart rate column (used if RR intervals not available)
        rr_column: Name of RR intervals column (in milliseconds)
        group_by: List of column names to group by (e.g., ['subject', 'Sol'])
    
    Returns:
        DataFrame with HRV metrics
    """
    # Check if RR intervals are directly available
    has_rr_intervals = rr_column in df.columns
    
    if not has_rr_intervals:
        if hr_column not in df.columns:
            print(f"❌ Neither '{rr_column}' nor '{hr_column}' found in data")
            print(f"Available columns: {list(df.columns)}")
            return pd.DataFrame()
        print(f"Using heart rate column '{hr_column}' to compute RR intervals")
    else:
        print(f"Using RR intervals directly from column '{rr_column}'")
    
    results = []
    
    def get_rr_intervals(data_subset: pd.DataFrame) -> np.ndarray:
        """Get RR intervals from either RR column or HR column."""
        if has_rr_intervals:
            rr = data_subset[rr_column].dropna().values
            # Filter physiologically plausible values
            rr = rr[(rr >= 300) & (rr <= 2000)]
            return rr
        else:
            return rr_from_hr(data_subset[hr_column])
    
    if group_by is None:
        # Analyze entire dataset
        print("Analyzing entire dataset...")
        rr_intervals = get_rr_intervals(df)
        if len(rr_intervals) >= 10:
            metrics = compute_comprehensive_hrv(rr_intervals, use_hrvanalysis=HAS_HRVANALYSIS)
            results.append(metrics)
        else:
            print(f"⚠️ Insufficient data: {len(rr_intervals)} RR intervals")
    else:
        # Group by specified columns
        available_groups = [col for col in group_by if col in df.columns]
        if not available_groups:
            print(f"⚠️ None of the grouping columns found. Analyzing entire dataset...")
            rr_intervals = get_rr_intervals(df)
            if len(rr_intervals) >= 10:
                metrics = compute_comprehensive_hrv(rr_intervals, use_hrvanalysis=HAS_HRVANALYSIS)
                results.append(metrics)
        else:
            print(f"Analyzing by: {', '.join(available_groups)}")
            for keys, group_df in df.groupby(available_groups):
                if isinstance(keys, tuple):
                    group_name = ' | '.join([f"{col}={val}" for col, val in zip(available_groups, keys)])
                else:
                    group_name = f"{available_groups[0]}={keys}"
                
                print(f"  Processing {group_name}...")
                rr_intervals = get_rr_intervals(group_df)
                
                if len(rr_intervals) >= 10:
                    metrics = compute_comprehensive_hrv(rr_intervals, use_hrvanalysis=HAS_HRVANALYSIS)
                    # Add grouping information
                    if isinstance(keys, tuple):
                        for col, val in zip(available_groups, keys):
                            metrics[col] = val
                    else:
                        metrics[available_groups[0]] = keys
                    results.append(metrics)
                else:
                    print(f"    ⚠️ Insufficient data: {len(rr_intervals)} RR intervals")
    
    if results:
        results_df = pd.DataFrame(results)
        print(f"\n✅ Analysis complete: {len(results_df)} segments analyzed")
        return results_df
    else:
        print("❌ No results generated")
        return pd.DataFrame()

# Perform analysis
# Modify group_by parameter based on your data structure
group_by_columns = []
if 'subject' in df.columns:
    group_by_columns.append('subject')
if 'Sol' in df.columns:
    group_by_columns.append('Sol')

hrv_results = analyze_hrv_dataframe(df, 
                                  hr_column='heart_rate [bpm]',
                                  group_by=group_by_columns if group_by_columns else None)

# Display results
if not hrv_results.empty:
    print("\nHRV Analysis Results:")
    print("=" * 80)
    display(hrv_results)
else:
    print("No results to display")


In [ ]:
# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================

def plot_rr_timeseries(rr_intervals: np.ndarray, title: str = "RR Interval Time Series"):
    """Plot RR interval time series."""
    if len(rr_intervals) == 0:
        print("No data to plot")
        return
    
    plt.figure(figsize=(14, 4))
    plt.plot(np.arange(len(rr_intervals)), rr_intervals / 1000.0, linewidth=0.7)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Beat index", fontsize=12)
    plt.ylabel("RR interval (s)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_psd(rr_intervals: np.ndarray, title: str = "Power Spectral Density", 
             sampling_rate: float = 4.0):
    """Plot power spectral density."""
    if len(rr_intervals) < 50:
        print("Insufficient data for PSD analysis")
        return
    
    try:
        # Interpolate RR intervals
        rr_seconds = rr_intervals / 1000.0
        r_peak_times = np.concatenate([[0], np.cumsum(rr_seconds)])
        rr_timestamps = (r_peak_times[:-1] + r_peak_times[1:]) / 2.0
        
        total_duration = r_peak_times[-1]
        time_regular = np.arange(0, total_duration, 1/sampling_rate)
        
        f_interp = interp1d(rr_timestamps, rr_intervals, 
                          kind='linear', bounds_error=False, fill_value='extrapolate')
        rr_interpolated = f_interp(time_regular)
        rr_detrended = signal.detrend(rr_interpolated)
        
        # Compute PSD
        freqs, psd = signal.welch(rr_detrended, fs=sampling_rate, 
                                nperseg=min(256, len(rr_detrended)//4))
        
        plt.figure(figsize=(10, 5))
        plt.semilogy(freqs, psd, lw=1.2)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.xlabel("Frequency (Hz)", fontsize=12)
        plt.ylabel("PSD (ms²/Hz)", fontsize=12)
        
        # Add frequency band markers
        plt.axvline(0.04, color='r', linestyle='--', alpha=0.5, label='LF band')
        plt.axvline(0.15, color='r', linestyle='--', alpha=0.5)
        plt.axvline(0.4, color='g', linestyle='--', alpha=0.5, label='HF band')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error plotting PSD: {e}")

def plot_poincare(rr_intervals: np.ndarray, title: str = "Poincaré Plot"):
    """Plot Poincaré plot."""
    if len(rr_intervals) < 2:
        print("Insufficient data for Poincaré plot")
        return
    
    rr1 = rr_intervals[:-1]
    rr2 = rr_intervals[1:]
    
    plt.figure(figsize=(8, 8))
    plt.scatter(rr1, rr2, s=8, alpha=0.6)
    
    # Add identity line
    lims = [min(rr1.min(), rr2.min()), max(rr1.max(), rr2.max())]
    plt.plot(lims, lims, "k--", alpha=0.6, label='Identity line')
    
    # Add SD1 and SD2 ellipses
    sd1 = np.std(rr2 - rr1) / np.sqrt(2)
    sd2 = np.std(rr2 + rr1) / np.sqrt(2)
    mean_rr = np.mean(rr_intervals)
    
    from matplotlib.patches import Ellipse
    ellipse = Ellipse((mean_rr, mean_rr), 2*sd1, 2*sd2, 
                     angle=45, alpha=0.2, color='red', label='SD1/SD2 ellipse')
    plt.gca().add_patch(ellipse)
    
    plt.xlabel("RRₙ (ms)", fontsize=12)
    plt.ylabel("RRₙ₊₁ (ms)", fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions defined")


In [ ]:
# Generate visualizations for a sample segment
if not hrv_results.empty and 'df' in locals():
    # Get RR intervals for visualization
    # Check if RR intervals are directly available
    if 'rr_intervals_ms' in df.columns:
        # Use RR intervals directly
        if group_by_columns:
            first_group_key = df[group_by_columns[0]].iloc[0] if group_by_columns else None
            if first_group_key is not None:
                sample_df = df[df[group_by_columns[0]] == first_group_key]
            else:
                sample_df = df
        else:
            sample_df = df
        sample_rr = sample_df['rr_intervals_ms'].dropna().values
        sample_rr = sample_rr[(sample_rr >= 300) & (sample_rr <= 2000)]
    else:
        # Convert from heart rate
        if group_by_columns:
            first_group = df.groupby(group_by_columns).first()
            if not first_group.empty:
                sample_rr = rr_from_hr(df[df[group_by_columns[0]] == df[group_by_columns[0]].iloc[0]]['heart_rate [bpm]'])
            else:
                sample_rr = rr_from_hr(df['heart_rate [bpm]'])
        else:
            sample_rr = rr_from_hr(df['heart_rate [bpm]'])
    
    if len(sample_rr) >= 10:
        print("Generating visualizations...")
        plot_rr_timeseries(sample_rr, "RR Interval Time Series - Sample Data")
        
        if len(sample_rr) >= 50:
            plot_psd(sample_rr, "Power Spectral Density - Sample Data")
        
        if len(sample_rr) >= 2:
            plot_poincare(sample_rr, "Poincaré Plot - Sample Data")
    else:
        print("Insufficient data for visualization")


## 5. Summary Statistics and Comparisons

Generate summary statistics and compare HRV metrics across groups.


In [ ]:
# =============================================================================
# SUMMARY STATISTICS AND COMPARISONS
# =============================================================================

if not hrv_results.empty:
    print("HRV Metrics Summary")
    print("=" * 80)
    
    # Key metrics to display
    key_metrics = ['mean_hr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 
                   'lf_hf_ratio', 'sd1', 'sd2', 'parasympathetic_index', 'sympathetic_index']
    
    available_metrics = [m for m in key_metrics if m in hrv_results.columns]
    
    if available_metrics:
        summary = hrv_results[available_metrics].describe()
        display(summary)
        
        # Group comparisons if grouping columns exist
        if group_by_columns:
            group_cols = [col for col in group_by_columns if col in hrv_results.columns]
            if group_cols:
                print(f"\n\nGrouped Summary Statistics")
                print("=" * 80)
                grouped_summary = hrv_results.groupby(group_cols)[available_metrics].agg(['mean', 'std', 'count'])
                display(grouped_summary)
    else:
        print("No key metrics available for summary")
        
    # Correlation matrix of key metrics
    if len(available_metrics) > 1:
        print(f"\n\nCorrelation Matrix of Key Metrics")
        print("=" * 80)
        corr_matrix = hrv_results[available_metrics].corr()
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                   square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title("HRV Metrics Correlation Matrix", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("No results available for summary statistics")


## 6. Export Results

Export HRV analysis results to CSV for further analysis or reporting.


In [ ]:
# Export results to CSV
if not hrv_results.empty:
    output_file = 'hrv_analysis_results.csv'
    hrv_results.to_csv(output_file, index=False)
    print(f"✅ Results exported to: {output_file}")
    print(f"   Total metrics: {len(hrv_results.columns)}")
    print(f"   Total segments: {len(hrv_results)}")
else:
    print("No results to export")


## Notes and References

### HRV Metrics Interpretation

**Time Domain Metrics:**
- **SDNN**: Standard deviation of NN intervals - overall HRV measure
- **RMSSD**: Root mean square of successive differences - short-term variability (parasympathetic)
- **pNN50**: Percentage of successive intervals differing by >50ms - parasympathetic indicator

**Frequency Domain Metrics:**
- **VLF (0.0033-0.04 Hz)**: Very low frequency power - thermoregulation, hormonal
- **LF (0.04-0.15 Hz)**: Low frequency power - baroreceptor activity, mixed ANS
- **HF (0.15-0.4 Hz)**: High frequency power - respiratory sinus arrhythmia, parasympathetic
- **LF/HF Ratio**: Sympathovagal balance indicator

**Nonlinear Metrics:**
- **SD1/SD2**: Poincaré plot measures - short-term and long-term variability
- **DFA α1/α2**: Detrended fluctuation analysis - fractal scaling properties

### References

1. Task Force of the European Society of Cardiology. (1996). Heart rate variability: standards of measurement, physiological interpretation, and clinical use. *Circulation*, 93(5), 1043-1065.

2. Shaffer, F., & Ginsberg, J. P. (2017). An overview of heart rate variability metrics and norms. *Frontiers in public health*, 5, 258.

3. Malik, M., et al. (1996). Heart rate variability: Standards of measurement, physiological interpretation, and clinical use. *European Heart Journal*, 17(3), 354-381.
